In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

import shap

In [ ]:
os.makedirs("/kaggle/working/models", exist_ok=True)
os.makedirs("/kaggle/working/reports", exist_ok=True)
os.makedirs("/kaggle/working/figures", exist_ok=True)

In [ ]:
df = pd.read_csv(
    "/kaggle/input/datasets/mystic00001/cleaned-dataset-csv/cleaned_dataset.csv"
)

print(df.shape)

df.head()

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print(
    "Unique Diseases:",
    df["diseases"].nunique()
)

In [ ]:
X = df.drop(
    columns=[
        "diseases",
        "disease_encoded"
    ]
)

y = df["disease_encoded"]

In [ ]:
def clean_cols(cols):

    cleaned = []

    for col in cols:

        col = str(col)

        col = re.sub(
            r"[^a-zA-Z0-9_]",
            "_",
            col
        )

        col = re.sub(
            r"_+",
            "_",
            col
        )

        cleaned.append(col)

    return cleaned

X.columns = clean_cols(X.columns)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [ ]:
def evaluate_model(
    model,
    X,
    y
):

    preds = model.predict(X)

    return {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(
            y,
            preds,
            average="weighted",
            zero_division=0
        ),
        "recall": recall_score(
            y,
            preds,
            average="weighted",
            zero_division=0
        ),
        "f1": f1_score(
            y,
            preds,
            average="weighted",
            zero_division=0
        )
    }

In [ ]:
dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(
    X_train,
    y_train
)

dt_results = evaluate_model(
    dt,
    X_val,
    y_val
)

dt_results

In [1]:
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    random_state=42,
    n_jobs=2
)

rf.fit(
    X_train,
    y_train
)

rf_results = evaluate_model(
    rf,
    X_val,
    y_val
)

rf_results

NameError: name 'RandomForestClassifier' is not defined

In [ ]:
xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_estimators=150,
    max_depth=8,
    learning_rate=0.1
)

xgb.fit(
    X_train,
    y_train
)

xgb_results = evaluate_model(
    xgb,
    X_val,
    y_val
)

xgb_results

In [ ]:
comparison = pd.DataFrame(
{
    "Decision Tree": dt_results,
    "Random Forest": rf_results,
    "XGBoost": xgb_results
}
).T

comparison.sort_values(
    "f1",
    ascending=False
)

In [ ]:
comparison.to_csv(
    "/kaggle/working/reports/model_comparison.csv"
)

In [ ]:
joblib.dump(
    dt,
    "/kaggle/working/models/decision_tree.pkl"
)

joblib.dump(
    rf,
    "/kaggle/working/models/random_forest.pkl"
)

joblib.dump(
    xgb,
    "/kaggle/working/models/xgboost.pkl"
)

joblib.dump(
    xgb,
    "/kaggle/working/models/best_model.pkl"
)

In [ ]:
probs = xgb.predict_proba(
    X_test
)

top3_preds = np.argsort(
    probs,
    axis=1
)[:, -3:]

correct = 0

y_test_np = y_test.to_numpy()

for i in range(len(y_test_np)):

    if y_test_np[i] in top3_preds[i]:
        correct += 1

top3_accuracy = (
    correct / len(y_test_np)
)

print(
    f"Top-3 Accuracy: {top3_accuracy:.4f}"
)

In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

importance.head(20)

In [ ]:
Best Model: XGBoost

Accuracy: 83.51%

F1 Score: 83.49%

Top-3 Accuracy: <fill after execution>

Reason for Selection:
XGBoost achieved the highest overall performance among all evaluated models and was selected as the final deployment model for the Streamlit application.